In [1]:
%pip install pandas numpy streamlit plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np

In [ ]:
df_pib = pd.read_csv("../data/raw/gdp-worldbank.csv")
df_pib.head()

,Entity,Code,Year,GDP
0,Afghanistan,AFG,2000,32567375969
1,Afghanistan,AFG,2001,29495629513
2,Afghanistan,AFG,2002,37931379899
3,Afghanistan,AFG,2003,41281584746
4,Afghanistan,AFG,2004,41865355064


In [9]:
df_pib.info()

<class 'pandas.DataFrame'>
RangeIndex: 7445 entries, 0 to 7444
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Entity  7445 non-null   str  
 1   Code    7445 non-null   str  
 2   Year    7445 non-null   int64
 3   GDP     7445 non-null   int64
dtypes: int64(2), str(2)
memory usage: 326.6 KB


In [10]:
df_pib.isnull().sum()

Entity    0
Code      0
Year      0
GDP       0
dtype: int64

In [11]:
df_pib.shape

(7445, 4)

In [12]:
print("Primer año:", df_pib.Year.min())
print("Último año:", df_pib.Year.max())

Primer año: 1990
Último año: 2025


In [13]:
df_pib.duplicated(subset=["Code", "Year"]).sum()

np.int64(0)

In [ ]:
df_territorial = pd.read_csv("../data/raw/territorial.csv")
df_consumo = pd.read_csv("../data/raw/consumo.csv")
df_poblacion = pd.read_csv("../data/raw/poblacion.csv")

In [16]:
df_territorial.head()

,Entity,Code,Year,Annual CO₂ emissions
0,Afghanistan,AFG,1949,14656.0
1,Afghanistan,AFG,1950,80608.0
2,Afghanistan,AFG,1951,87936.0
3,Afghanistan,AFG,1952,87936.0
4,Afghanistan,AFG,1953,98928.0


In [17]:
df_consumo.head()

,Entity,Code,Year,Annual consumption-based CO₂ emissions
0,Africa,OWID_AFR,1990,653389200.0
1,Africa,OWID_AFR,1991,670270500.0
2,Africa,OWID_AFR,1992,667447550.0
3,Africa,OWID_AFR,1993,701330750.0
4,Africa,OWID_AFR,1994,775572700.0


In [18]:
df_poblacion.head()

,Entity,Code,Year,Population
0,Afghanistan,AFG,-10000,14737
1,Afghanistan,AFG,-9000,20405
2,Afghanistan,AFG,-8000,28253
3,Afghanistan,AFG,-7000,39120
4,Afghanistan,AFG,-6000,54166


In [19]:
df_pib = df_pib.rename(columns={"Entity":"pais", "Code":"codigo", "Year":"anio", "GDP":"pib"})

df_territorial = df_territorial.rename(columns={"Entity":"pais", "Code":"codigo", "Year":"anio", "Annual CO₂ emissions":"co2_territorial"})

df_consumo = df_consumo.rename(columns={"Entity":"pais", "Code":"codigo", "Year":"anio", "Annual consumption-based CO₂ emissions":"co2_consumo"})

df_poblacion = df_poblacion.rename(columns={"Entity":"pais", "Code":"codigo", "Year":"anio", "Population":"poblacion"})

In [20]:
print("PIB:", df_pib.shape)
print("Territorial:", df_territorial.shape)
print("Consumo:", df_consumo.shape)
print("Población:", df_poblacion.shape)

PIB: (7445, 4)
Territorial: (30219, 4)
Consumo: (5202, 4)
Población: (58824, 4)


In [21]:
print("PIB:", df_pib.anio.min(), "-", df_pib.anio.max())
print("Territorial:", df_territorial.anio.min(), "-", df_territorial.anio.max())
print("Consumo:", df_consumo.anio.min(), "-", df_consumo.anio.max())
print("Población:", df_poblacion.anio.min(), "-", df_poblacion.anio.max())

PIB: 1990 - 2025
Territorial: 1750 - 2025
Consumo: 1990 - 2025
Población: -10000 - 2023


In [22]:
df_pib = df_pib[df_pib.codigo.str.fullmatch("[A-Z]{3}", na=False)].copy()

df_territorial = df_territorial[df_territorial.codigo.str.fullmatch("[A-Z]{3}", na=False)].copy()

df_consumo = df_consumo[df_consumo.codigo.str.fullmatch("[A-Z]{3}", na=False)].copy()

df_poblacion = df_poblacion[df_poblacion.codigo.str.fullmatch("[A-Z]{3}", na=False)].copy()

In [23]:
print("Duplicados PIB:", df_pib.duplicated(subset=["codigo", "anio"]).sum())

print("Duplicados territorial:", df_territorial.duplicated(subset=["codigo", "anio"]).sum())

print("Duplicados consumo:", df_consumo.duplicated(subset=["codigo", "anio"]).sum())

print("Duplicados población:", df_poblacion.duplicated(subset=["codigo", "anio"]).sum())

Duplicados PIB: 0
Duplicados territorial: 0
Duplicados consumo: 0
Duplicados población: 0


In [24]:
pib_info = df_pib[["pais", "codigo", "anio", "pib"]]

territorial_info = df_territorial[["codigo", "anio", "co2_territorial"]]

consumo_info = df_consumo[["codigo", "anio", "co2_consumo"]]

poblacion_info = df_poblacion[["codigo", "anio", "poblacion"]]

In [25]:
df_union1 = pib_info.merge(territorial_info, on=["codigo", "anio"], how="left", validate="1:1")

df_union2 = df_union1.merge(consumo_info, on=["codigo", "anio"], how="left", validate="1:1")

df_final = df_union2.merge(poblacion_info, on=["codigo", "anio"], how="left", validate="1:1")

df_final.head()

,pais,codigo,anio,pib,co2_territorial,co2_consumo,poblacion
0,Afghanistan,AFG,2000,32567375969,1065448.0,NaN,20130334.0
1,Afghanistan,AFG,2001,29495629513,1083754.0,NaN,20284303.0
2,Afghanistan,AFG,2002,37931379899,1366645.0,NaN,21378123.0
3,Afghanistan,AFG,2003,41281584746,1585252.0,NaN,22733053.0
4,Afghanistan,AFG,2004,41865355064,1259315.0,NaN,23560656.0


In [26]:
print("PIB original:", pib_info.shape)
print("Primera unión:", df_union1.shape)
print("Segunda unión:", df_union2.shape)
print("Unión final:", df_final.shape)

print("Duplicados:", df_final.duplicated(subset=["codigo", "anio"]).sum())

PIB original: (6959, 4)
Primera unión: (6959, 5)
Segunda unión: (6959, 6)
Unión final: (6959, 7)
Duplicados: 0


In [27]:
df_final.isna().sum()

pais                  0
codigo                0
anio                  0
pib                   0
co2_territorial     112
co2_consumo        2845
poblacion           378
dtype: int64

In [28]:
union_inner = pib_info.merge(territorial_info, on=["codigo", "anio"], how="inner", validate="1:1")

union_outer = pib_info.merge(territorial_info, on=["codigo", "anio"], how="outer", validate="1:1")

comparacion = pd.DataFrame({"tipo_union":["inner", "left", "outer"], "numero_filas":[union_inner.shape[0], df_union1.shape[0], union_outer.shape[0]]})

comparacion

,tipo_union,numero_filas
0,inner,6847
1,left,6959
2,outer,24221


In [29]:
print("Registros antes de limpiar:", df_final.shape[0])

df_final = df_final.dropna(subset=["pib", "co2_territorial", "co2_consumo", "poblacion"]).copy()

df_final = df_final[(df_final.pib > 0) & (df_final.poblacion > 0)].copy()

print("Registros después de limpiar:", df_final.shape[0])

Registros antes de limpiar: 6959
Registros después de limpiar: 3995


In [30]:
df_final["territorial_per_capita"] = df_final.co2_territorial / df_final.poblacion

df_final["consumo_per_capita"] = df_final.co2_consumo / df_final.poblacion

df_final["intensidad_territorial"] = df_final.co2_territorial / df_final.pib * 1000

df_final["intensidad_consumo"] = df_final.co2_consumo / df_final.pib * 1000

df_final["brecha_emisiones"] = df_final.co2_consumo - df_final.co2_territorial

In [31]:
df_final = df_final.sort_values(["pais", "anio"]).reset_index(drop=True)

df_final.head()

,pais,codigo,anio,pib,co2_territorial,co2_consumo,poblacion,territorial_per_capita,consumo_per_capita,intensidad_territorial,intensidad_consumo,brecha_emisiones
0,Albania,ALB,1990,18275914282,5531896.0,5588138.5,3277965.0,1.687601,1.704758,0.302688,0.305765,56242.5
1,Albania,ALB,1991,13158266876,4285552.0,4614414.0,3282608.0,1.305533,1.405716,0.325693,0.350686,328862.0
2,Albania,ALB,1992,12212567642,2515386.0,2924834.8,3282506.0,0.766301,0.891037,0.205967,0.239494,409448.8
3,Albania,ALB,1993,13380017258,2336095.0,2674717.5,3277831.0,0.712695,0.816002,0.174596,0.199904,338622.5
4,Albania,ALB,1994,14490942242,1969568.0,2408765.0,3269417.0,0.602422,0.736757,0.135917,0.166226,439197.0


In [32]:
print("Cantidad de países:", df_final.codigo.nunique())
print("Cantidad de registros:", df_final.shape[0])
print("Primer año:", df_final.anio.min())
print("Último año:", df_final.anio.max())

df_final.groupby("anio")["codigo"].nunique()

Cantidad de países: 118
Cantidad de registros: 3995
Primer año: 1990
Último año: 2023


anio
1990    116
1991    117
1992    117
1993    117
1994    117
1995    117
1996    117
1997    117
1998    117
1999    117
2000    117
2001    117
2002    117
2003    118
2004    118
2005    118
2006    117
2007    118
2008    117
2009    118
2010    118
2011    118
2012    118
2013    118
2014    118
2015    118
2016    117
2017    118
2018    118
2019    118
2020    118
2021    118
2022    118
2023    118
Name: codigo, dtype: int64

In [ ]:
df_final.to_csv("../data/processed/datos_integrados.csv", index=False)